## Results

In [13]:
%run ../utils/sampling.py
import pandas as pd

product_info_df = pd.read_csv('../data/cleaned/product-features.csv')
departments = pd.read_csv('../dataset/departments.csv')
products = pd.read_csv('../dataset/products.csv')

full_products_df = product_info_df.merge(products, on='product_id', how='left')

product_names_to_search = [
    "Coke",
    "Bananas",
    "Organic Whole Milk",
    "Plain Bagels",
    "Spaghetti"
]

top_ids = get_top_product_ids(full_products_df, product_names_to_search)
print(top_ids)

found_products = search_products(full_products_df, "Coke")
print(found_products.head(1))

{'Coke': {'query': 'Coke', 'matched_name': 'Diet Coke', 'product_id': 43631, 'order_penetration_pct': 0.2090284098225933}, 'Bananas': {'query': 'Bananas', 'matched_name': 'Banana', 'product_id': 24852, 'order_penetration_pct': 14.69933191782944}, 'Organic Whole Milk': {'query': 'Organic Whole Milk', 'matched_name': 'Organic Whole Milk', 'product_id': 27845, 'order_penetration_pct': 4.289592686991776}, 'Plain Bagels': {'query': 'Plain Bagels', 'matched_name': 'Plain Bagels', 'product_id': 20738, 'order_penetration_pct': 0.3195770658507922}, 'Spaghetti': {'query': 'Spaghetti', 'matched_name': 'Spaghetti', 'product_id': 32734, 'order_penetration_pct': 0.49186997686379}}
   product_name  product_id  order_penetration_pct
0  Coke Classic       16696               0.335068


In [14]:
product_ids = [
    info["product_id"] 
    for info in top_ids.values() 
    if info is not None
]

product_ids.append(16696)
print(product_ids)
sampled_products_df = pd.DataFrame(product_ids, columns=['product_id'])
sampled_products_df.to_csv("../results3/sampled-products.csv", index=None)

[43631, 24852, 27845, 20738, 32734, 16696]


In [15]:
%run ../utils/pairwise.py

# Get product_ids as a series
products = product_ids.copy()
product_df = pd.read_csv('../data/cleaned/product-info-full.csv')
orders_full_df = pd.read_csv('../dataset/order_products__prior.csv')

order_product_df = orders_full_df[['order_id', 'product_id']]

# Compute pairwise probabilties for focus products
compute_pairwise_probabilities_sample(order_product_df, 
    products,
    output_csv="../results3/products-pairwise.csv",
    batch_size=1000
)

100%|██████████| 1/1 [00:04<00:00,  4.41s/it]

Completed computation. Saved to ../results3/products-pairwise.csv


In [16]:
%run ../utils/substitutes.py

dept3_df = pd.read_csv('../data/cleaned/pairwise-dept3.csv')
dept4_df = pd.read_csv('../data/cleaned/pairwise-dept4.csv')
dept7_df = pd.read_csv('../data/cleaned/pairwise-dept7.csv')
dept9_df = pd.read_csv('../data/cleaned/pairwise-dept9.csv')
dept16_df = pd.read_csv('../data/cleaned/pairwise-dept16.csv')

pairwise_df = pd.concat([dept3_df, dept4_df, dept7_df, dept9_df, dept16_df], ignore_index=True)

similarity_df = pd.read_csv('../data/cleaned/product-similiarity.csv')

compute_sub_score_by_dept(
    product_df,
    pairwise_df,
    products,
    similarity_df,
    "../results3/raw-substitutes.csv")

Completed substitute calculations. Saved to ../results3/raw-substitutes.csv


In [19]:
%run ../utils/substitutes.py

substitutes_df = pd.read_csv("../results3/raw-substitutes.csv")

# Find the best substitution score threshold which will identify a product as a true substitute
best_threshold = find_best_threshold(substitutes_df)

# Add a new column 'identified_substitute' based on the threshold
substitutes_df['identified_substitute'] = substitutes_df['score'] >= best_threshold

# Compute transferability %
results = compute_transferability(orders_full_df, substitutes_df, top_n=10)
results.to_csv("../results3/substitutes-transfer.csv", index=False)

 Best threshold determined as: 0.461 and adjusted to: 0.47651687847569185



In [20]:
%run ../utils/results.py

product_info = pd.read_csv('../dataset/products.csv')
aisle_info = pd.read_csv('../dataset/aisles.csv')
# Select only product_id + product_name from info DF
df_info_min = product_info[["product_id", "product_name"]].copy()

# 1. Merge to add product_name
df_merged = results.merge(
    df_info_min,
    on="product_id",
    how="left"
)

df_info = product_info[["product_id", "product_name", "aisle_id"]].copy()
aisle_info = aisle_info[["aisle_id", "aisle"]].copy()

df_info = df_info.merge(aisle_info, on="aisle_id", how="left")

# 2. Prepare a renamed version for substitute names
df_info_sub = df_info.rename(
    columns={
        "product_id": "substitute_id",
        "product_name": "sub_name"
    }
)

# 3. Merge to add sub_name
df_merged = df_merged.merge(
    df_info_sub,
    on="substitute_id",
    how="left"
)

show_all_substitute_tables(df_merged)


Product: Coke Classic  (ID: 16696)


,sub_name,transferability_pct,aisle
0,Fridge Pack Cola,0.125666,soft drinks
1,Ginger Ale,0.100465,soft drinks
2,Cola,0.093925,soft drinks
3,Root Beer,0.066454,soft drinks
4,Classic Soda,0.062034,soft drinks
5,Diet Ginger Ale All Natural Soda,0.057930,soft drinks
6,Coke Zero,0.053436,soft drinks
7,Original Citrus Sparkling Flavored Soda,0.048376,soft drinks
8,"Ginger Root Beer, Naturally Flavored Zero Calo...",0.045216,soft drinks
9,Ginger Ale Soda,0.042925,soft drinks



Product: Plain Bagels  (ID: 20738)


,sub_name,transferability_pct,aisle
0,Plain Pre-Sliced Bagels,0.126002,breakfast bakery
1,Plain Mini Bagels,0.113499,breakfast bakery
2,Everything Bagels,0.094180,breakfast bakery
3,Whole Wheat English Muffins,0.086830,breakfast bakery
4,Plain Bagelettes,0.079532,breakfast bakery
5,Flourless Sprouted Whole Grain 7-Sprouted Gra...,0.051232,breakfast bakery
6,Onion Bagels,0.043757,breakfast bakery
7,Bagels Plain Presliced,0.040700,breakfast bakery
8,Gluten Free Double Chocolate Muffins,0.035620,breakfast bakery
9,100% Whole Wheat English Muffin 6 Ct,0.034543,breakfast bakery



Product: Banana  (ID: 24852)


...

,sub_name,transferability_pct,aisle
0,2% Reduced Fat Milk,0.070946,milk
1,Whole Milk,0.069496,milk
2,Organic Whole Milk,0.068492,milk
3,Fat Free Milk,0.061651,milk
4,Organic Milk,0.061470,milk
5,Whole Organic Omega 3 Milk,0.060415,milk
6,Organic 2% Milk,0.055318,milk
7,Organic Grade A Raw Whole Milk,0.054716,milk
8,Organic Lactose Free Whole Milk,0.051266,soy lactosefree
9,Organic Plain Whole Milk Yogurt,0.051127,yogurt



Product: Spaghetti  (ID: 32734)


,sub_name,transferability_pct,aisle
0,Spaghetti Pasta,0.134408,dry pasta
1,Organic Penne Rigate,0.106179,dry pasta
2,Organic Orzo,0.083488,dry pasta
3,Organic Brown Rice Pasta Spirals,0.054049,dry pasta
4,Pasta Joy Ready Organic Brown Rice Pasta Penne,0.048754,dry pasta
5,ProteinPLUS Multigrain Penne Pasta,0.042879,dry pasta
6,Gluten Free Brown Rice Penne Rigate Pasta,0.042149,dry pasta
7,Chickpeas! Penne Pasta,0.040807,dry pasta
8,Organic Gluten Free Elbow Pasta,0.040703,dry pasta
9,Organic Linguine,0.039108,dry pasta



Product: Diet Coke  (ID: 43631)


,sub_name,transferability_pct,aisle
0,Soda,0.471083,soft drinks
1,Diet Ginger Ale All Natural Soda,0.096618,soft drinks
2,Original Citrus Sparkling Flavored Soda,0.079447,soft drinks
3,"Ginger Root Beer, Naturally Flavored Zero Calo...",0.073036,soft drinks
4,Diet Cola,0.059744,soft drinks
5,Diet Ginger Ale,0.047743,soft drinks
6,Coke,0.039585,soft drinks
7,Zero Calorie Grape Soda,0.038620,soft drinks
8,Slim Can Cola,0.026809,soft drinks
9,Vanilla Coke Zero,0.023836,soft drinks


In [23]:
%run ../utils/complements.py

sample_df = pd.read_csv("../results3/sampled-products.csv")
pairwise_df = pd.read_csv("../results3/products-pairwise.csv")

num_orders = order_product_df['order_id'].nunique()
min_co_occurrences = 5  # at least 5 orders
min_pij = min_co_occurrences / num_orders

lift_df = compute_lift(sample_df['product_id'].to_list(), pairwise_df,min_pij=min_pij, total_orders=num_orders)
lift_df.to_csv("../results3/lift.csv", index=False)

complements_df = compute_hybrid_score(lift_df, sample_df['product_id'].to_list(), top_n=10)
complements_df.to_csv("../results3/complements.csv", index=False)

Computing lift: 100%|██████████| 6/6 [00:01<00:00,  4.40it/s]


In [30]:
%run ../utils/complements.py

cii_df = compute_complement_impact_index(complements_df, pairwise_df)
cii_df.to_csv("../results3/complements-cii.csv", index=False)

In [ ]:
%run ../utils/results.py

product_info = pd.read_csv('../dataset/products.csv')
aisle_info = pd.read_csv('../dataset/aisles.csv')
# Select only product_id + product_name from info DF
df_info_min = product_info[["product_id", "product_name"]].copy()

# 1. Merge to add product_name
df_merged = complements_df.merge(
    df_info_min,
    on="product_id",
    how="left"
)

df_info = product_info[["product_id", "product_name", "aisle_id"]].copy()
aisle_info = aisle_info[["aisle_id", "aisle"]].copy()

df_info = df_info.merge(aisle_info, on="aisle_id", how="left")

# 2. Prepare a renamed version for substitute names
df_info_comp = df_info.rename(
    columns={
        "product_id": "complement_id",
        "product_name": "comp_name"
    }
)

# 3. Merge to add sub_name
df_merged = df_merged.merge(
    df_info_comp,
    on="complement_id",
    how="left"
)

show_all_complements_tables(df_merged)

Index(['product_id', 'complement_id', 'lift', 'b_complementarity',
       'hybrid_score', 'product_name', 'comp_name', 'aisle_id', 'aisle'],
      dtype='object')

Product: Coke Classic  (ID: 16696)


,comp_name,hybrid_score,aisle
0,Lemon Lime Soda Caffeine Free,0.820062,soft drinks
1,Classic Caffeine Free Soda,0.688852,soft drinks
2,Vanilla Coke,0.550834,soft drinks
3,Ginger Soda,0.491417,soft drinks
4,Chocolate Favorites Fun Size Variety Pack,0.437364,candy chocolate
5,Seasoned Black Cherry Barbecue Pork jerky,0.416736,popcorn jerky
6,Multi-Grain English Muffins,0.373669,breakfast bakery
7,Deluxe Mixed Nuts,0.347049,nuts seeds dried fruit
8,Roasted Garlic Hummus with Pretzels,0.333987,fresh dips tapenades
9,Miniatures Assortment Party Bag,0.322294,candy chocolate



Product: Plain Bagels  (ID: 20738)


,comp_name,hybrid_score,aisle
0,Sliced Pepper Jack Cheese,0.559833,packaged cheese
1,Hearty & Delicious 100% Whole Wheat Bread,0.502512,bread
2,Soft Cream Cheese,0.428556,other creams cheeses
3,Wheat Sandwich Bread,0.375519,bread
4,Double Chocolate Muffins,0.343234,breakfast bakery
5,Clean Burst Liquid Laundry Detergent,0.338362,laundry
6,Extra Sharp White Cheddar Sticks,0.327945,packaged cheese
7,Deluxe Bagels Onion,0.269454,breakfast bakery
8,Regular Cream Cheese Spread,0.259858,other creams cheeses
9,Halloumi Cheese,0.258872,specialty cheeses



Product: Banana  (ID: 24852)


...

,comp_name,hybrid_score,aisle
0,Organic Superfoods Carrot Rice Cakes,0.068897,baby food formula
1,Toddler Cheddar & Leeks Multigrain Wheels Orga...,0.068363,baby food formula
2,Crunchin' 123 Sesame Street Veggie Crackers,0.068086,baby food formula
3,"Crunchin' Grahams, Honey Sticks, 123 Sesame St...",0.058939,baby food formula
4,Organic Pineapple Orange Banana Fruit Yogurt S...,0.055348,baby food formula
5,"Organic Pears, Apples, Peaches, Pumpkin + Cinn...",0.054145,baby food formula
6,Organic Amaze Mint Baby Food,0.053530,baby food formula
7,Smoothie Fruits Squished The Purple One Over 6...,0.052175,baby food formula
8,"Smoothie Fruits, Squished, The Green One, Over...",0.051877,baby food formula
9,"Fruit Snack, 100% Pure, Organic, Peach and Apple",0.050774,baby food formula



Product: Spaghetti  (ID: 32734)


,comp_name,hybrid_score,aisle
0,Shells & White Cheddar,0.172384,instant foods
1,Parmesan Beef Meatballs,0.153591,meat counter
2,Homestyle Tart Cherry Lemonade Drink,0.134526,juice nectars
3,Rao's Homemade Roasted Garlic Sauce,0.120286,pasta sauce
4,Ground Sausage Style Veggie Protein,0.118726,tofu meat alternatives
5,Roasted Garlic Alfredo Pasta Sauce,0.118265,pasta sauce
6,Old World Style Organic Traditional Pasta Sauce,0.117315,pasta sauce
7,Fresh Tilapia Fillets,0.117026,seafood counter
8,Tomato and Basil Bombolina Pasta Sauce,0.112244,pasta sauce
9,Chunky Tomato Garlic & Onion Pasta Sauce,0.111572,pasta sauce



Product: Diet Coke  (ID: 43631)


,comp_name,hybrid_score,aisle
0,Pepsi,1.000000,soft drinks
1,Soft And Strong Double Roll Bath Tissue,0.710828,paper goods
2,Peach Citrus Soda,0.605229,soft drinks
3,Nuggets Assortment,0.564574,candy chocolate
4,Paper Towels White w/ Thirst Pockets,0.519153,paper goods
5,Light Boston Cream Pie Yogurt,0.511912,yogurt
6,Protein Chewy Bar Peanut Butter Dark Chocolate...,0.492086,fruit vegetable snacks
7,Miniatures Assortment Party Bag,0.480176,candy chocolate
8,Juice Drink Variety Pack,0.474706,juice nectars
9,Caffeine Free Diet Coke,0.464558,soft drinks


In [28]:
def analyze_product_complements(complements_df, pairwise_df, product_lookup, focus_product_id, top_n=20):
    """
    Creates a detailed table for a single product's complements.

    Parameters
    ----------
    complements_df : pd.DataFrame
        Columns: ['product_id','complement_id','lift','b_complementarity','hybrid_score']
    product_lookup : pd.DataFrame
        Columns: ['product_id','product_name','aisle']
    focus_product_id : int
        Product to analyze.
    top_n : int
        Number of top complements to include.

    Returns
    -------
    pd.DataFrame
    """
    df = complements_df[complements_df['product_id'] == focus_product_id].copy()

    df = df.merge(
        pairwise_df,
        left_on=['product_id', 'complement_id'],
        right_on=['product_i', 'product_j'],
        how='left'
    )


    # Merge aisle and name info for complements
    df = df.merge(
        product_lookup[['product_id','product_name','aisle']],
        left_on='complement_id', right_on='product_id',
        how='left',
        suffixes=('', '_comp')
    )

    # Merge aisle info for focus product
    focus_aisle = product_lookup.loc[product_lookup['product_id']==focus_product_id, 'aisle'].values[0]
    df['same_aisle'] = df['aisle'] == focus_aisle

    # Sort by hybrid score
    df = df.sort_values('hybrid_score', ascending=False).head(top_n)

    # Select relevant columns
    df = df[['product_id','complement_id','product_name','P_i','P_j','P_ij','lift','b_complementarity','hybrid_score','aisle','same_aisle']]
    df = df.rename(columns={'product_name':'comp_name'})

    return df

# Example usage:
focus_product_id = 16696  # Coke Classic
product_lookup = product_info[["product_id", "product_name", "aisle_id"]].copy()
product_lookup_full = product_lookup.merge(aisle_info, on="aisle_id", how="left")
detailed_table = analyze_product_complements(complements_df, pairwise_df, product_lookup_full, focus_product_id, top_n=20)
display(detailed_table)


,product_id,complement_id,comp_name,P_i,P_j,P_ij,lift,b_complementarity,hybrid_score,aisle,same_aisle
0,16696,11571,Lemon Lime Soda Caffeine Free,0.003351,0.000162,0.000033,61.175973,0.967833,0.820062,soft drinks,True
1,16696,43094,Classic Caffeine Free Soda,0.003351,0.000112,0.000019,51.542426,0.961936,0.688852,soft drinks,True
2,16696,17981,Vanilla Coke,0.003351,0.000119,0.000016,41.407606,0.952839,0.550834,soft drinks,True
3,16696,21650,Ginger Soda,0.003351,0.000133,0.000016,37.043806,0.947429,0.491417,soft drinks,True
4,16696,18790,Chocolate Favorites Fun Size Variety Pack,0.003351,0.000118,0.000013,33.073312,0.941303,0.437364,candy chocolate,False
5,16696,47787,Seasoned Black Cherry Barbecue Pork jerky,0.003351,0.000103,0.000011,31.557869,0.938571,0.416736,popcorn jerky,False
6,16696,32315,Multi-Grain English Muffins,0.003351,0.000147,0.000014,28.393504,0.931958,0.373669,breakfast bakery,False
7,16696,11331,Deluxe Mixed Nuts,0.003351,0.000316,0.000028,26.437258,0.927106,0.347049,nuts seeds dried fruit,False
8,16696,32636,Roasted Garlic Hummus with Pretzels,0.003351,0.000102,0.000009,25.477206,0.924463,0.333987,fresh dips tapenades,False
9,16696,22166,Miniatures Assortment Party Bag,0.003351,0.000275,0.000023,24.617684,0.921929,0.322294,candy chocolate,False


In [ ]:
%run ../utils/complements.py

pairwise_df = pd.read_csv("../results3/products-pairwise.csv")
complements_df = pd.read_csv("../results3/complements-cii.csv")
network_df = compute_network_enhanced_impact(complements_df, pairwise_df)

In [39]:
network_df.to_csv("../results3/complements-cii-enhanced.csv", index=False)

In [34]:
import pandas as pd
from scipy.stats import kendalltau

def evaluate_impact_methods(
    original_df,
    enhanced_df,
    k_values=[3, 5, 10],
    verbose=True
):
    """
    Evaluates the difference between original and network-enhanced impact indexes.

    Parameters
    ----------
    original_df : DataFrame
        Must contain columns: product_i, product_j, impact_index_j
    enhanced_df : DataFrame
        Must contain columns: product_i, product_j, impact_index_j
    k_values : list
        List of k values for top-k overlap
    verbose : bool
        If True, prints a readable report

    Returns
    -------
    results_df : DataFrame
        Per-product evaluation metrics.
    summary : dict
        Aggregated metrics.
    """

    # Ensure sorted
    orig = original_df.sort_values(["product_i", "impact_index_j"], ascending=[True, False])
    enh = enhanced_df.sort_values(["product_i", "impact_index_j"], ascending=[True, False])

    all_products = sorted(orig["product_i"].unique())

    results = []

    for pid in all_products:
        orig_sub = orig[orig["product_i"] == pid]
        enh_sub = enh[enh["product_i"] == pid]

        # Both must have the same complement set
        merged = orig_sub[["product_j", "impact_index_j"]].merge(
            enh_sub[["product_j", "impact_index_j"]],
            on="product_j",
            suffixes=("_orig", "_enh")
        )

        if len(merged) < 2:
            continue  # Need at least 2 values for correlation

        # Ranks for Kendall Tau
        tau, _ = kendalltau(
            merged["impact_index_j_orig"].rank(ascending=False),
            merged["impact_index_j_enh"].rank(ascending=False)
        )

        # Top-k overlap
        topk_results = {}
        for k in k_values:
            orig_topk = set(orig_sub.head(k)["product_j"])
            enh_topk  = set(enh_sub.head(k)["product_j"])
            overlap = len(orig_topk & enh_topk) / max(1, len(orig_topk))
            topk_results[f"top{k}_overlap"] = overlap

        results.append({
            "product_i": pid,
            "kendall_tau": tau,
            **topk_results
        })

    results_df = pd.DataFrame(results)

    # Summary metrics
    summary = {
        "kendall_tau_mean": results_df["kendall_tau"].mean(),
        "kendall_tau_median": results_df["kendall_tau"].median(),
        "kendall_tau_min": results_df["kendall_tau"].min(),
        "kendall_tau_max": results_df["kendall_tau"].max(),
    }

    for k in k_values:
        summary[f"top{k}_overlap_mean"] = results_df[f"top{k}_overlap"].mean()

    if verbose:
        print("\n==============================")
        print(" Network Impact Evaluation Report")
        print("==============================\n")

        print("🔹 Ranking Similarity (Kendall Tau)")
        print(f"  Mean:   {summary['kendall_tau_mean']:.4f}")
        print(f"  Median: {summary['kendall_tau_median']:.4f}")
        print(f"  Min:    {summary['kendall_tau_min']:.4f}")
        print(f"  Max:    {summary['kendall_tau_max']:.4f}\n")

        print("🔹 Top-K Overlap")
        for k in k_values:
            print(f"  Top-{k} mean overlap: {summary[f'top{k}_overlap_mean']:.3f}")
        print()

        print("✔️ Evaluation complete. Paste your results_df or summary here and I can help interpret it.")

    return results_df, summary

results_df, summary = evaluate_impact_methods(complements_df, network_df)



 Network Impact Evaluation Report

🔹 Ranking Similarity (Kendall Tau)
  Mean:   0.9185
  Median: 1.0000
  Min:    0.7333
  Max:    1.0000

🔹 Top-K Overlap
  Top-3 mean overlap: 0.944
  Top-5 mean overlap: 0.933
  Top-10 mean overlap: 1.000

✔️ Evaluation complete. Paste your results_df or summary here and I can help interpret it.


In [35]:
# Sort from largest to smallest change
largest_changes = results_df.sort_values("kendall_tau").head(10)
largest_changes


,product_i,kendall_tau,top3_overlap,top5_overlap,top10_overlap
0,16696,0.733333,1.000000,0.8,1.0
5,43631,0.777778,0.666667,0.8,1.0
1,20738,1.000000,1.000000,1.0,1.0
2,24852,1.000000,1.000000,1.0,1.0
3,27845,1.000000,1.000000,1.0,1.0
4,32734,1.000000,1.000000,1.0,1.0


In [37]:
product_list = largest_changes["product_i"].tolist()

for pid in product_list:
    print("\n===============================")
    print(f"Product {pid} — Original vs Enhanced")
    print("===============================\n")

    orig_rank = complements_df[complements_df["product_i"] == pid][["product_j", "impact_index_j"]].sort_values("impact_index_j", ascending=False).head(10)
    enh_rank  = network_df[network_df["product_i"] == pid][["product_j", "impact_index_j"]].sort_values("impact_index_j", ascending=False).head(10)

    print("🔹 Original Top 10:")
    print(orig_rank)

    print("\n🔹 Enhanced Top 10:")
    print(enh_rank)
    print("\n")



Product 16696 — Original vs Enhanced

🔹 Original Top 10:
   product_j  impact_index_j
0      11571        0.061494
1      43094        0.051811
2      17981        0.041623
3      21650        0.037237
4      18790        0.033245
5      47787        0.031722
6      32315        0.028541
7      11331        0.026575
8      32636        0.025610
9      22166        0.024746

🔹 Enhanced Top 10:
   product_j  impact_index_j
0      11571        1.190587
1      43094        0.853358
2      17981        0.559002
9      22166        0.476574
3      21650        0.451340
4      18790        0.363361
5      47787        0.332289
6      32315        0.271874
7      11331        0.237546
8      32636        0.221543



Product 43631 — Original vs Enhanced

🔹 Original Top 10:
    product_j  impact_index_j
50      22249        0.046646
51      45192        0.033333
52       3499        0.028471
53      28748        0.026599
54       7663        0.024507
55      27238        0.024174
56      15258 

In [38]:
summary

{'kendall_tau_mean': 0.9185185185185184,
 'kendall_tau_median': 0.9999999999999999,
 'kendall_tau_min': 0.7333333333333333,
 'kendall_tau_max': 0.9999999999999999,
 'top3_overlap_mean': 0.9444444444444445,
 'top5_overlap_mean': 0.9333333333333332,
 'top10_overlap_mean': 1.0}

In [40]:
import pandas as pd
import numpy as np

def compute_total_impact(
    pairwise_impact_df,
    penetration_df,
    product_i_col="product_id",
    product_j_col="complement_id",
    impact_col="impact_index_j",
    penetration_col="order_penetration_pct",   # if percent (0-100). function will detect and convert
    method="topk_weighted",              # options: "weighted_sum", "topk_weighted", "normalized"
    top_k=5,
    normalize=False,
    min_penetration=1e-6,
    min_impact=0.0,
    return_details=False
):
    """
    Compute total impact per product i using penetration of complements (j).

    Inputs:
    - pairwise_impact_df: DataFrame with columns [product_i, product_j, impact_index_j, ...]
      (impact_col should already be computed, either pairwise or network-enhanced)
    - penetration_df: DataFrame or Series with product j penetration info.
      If DataFrame, must have columns [product_id, penetration_col].
      If Series, index=product_id, values=penetration (pct or fraction).
    - method: "weighted_sum" (all j), "topk_weighted" (only top_k by impact), "normalized" (divides by sum of weights)
    - top_k: how many top complements to include (only used for topk_weighted)
    - normalize: if True, returns total impact normalized across all i to 0-1 by dividing by max
    - min_penetration: floor to avoid dividing by zero / tiny weights
    - min_impact: filter small pairwise impacts
    - return_details: if True returns merged long-form df with pair-level contributions

    Returns:
    - totals_df: DataFrame with columns ['product_i', 'total_impact', 'total_weight', ...]
    - (optionally) details_df: merged pair-level contributions per (i,j)
    """

    # normalize penetration input to series indexed by product id
    if isinstance(penetration_df, pd.DataFrame):
        if 'product_id' in penetration_df.columns:
            pen = penetration_df.set_index('product_id')[penetration_col].astype(float)
        else:
            # assume first column is product id
            pen = penetration_df.set_index(penetration_df.columns[0])[penetration_col].astype(float)
    elif isinstance(penetration_df, pd.Series):
        pen = penetration_df.astype(float)
    else:
        raise ValueError("penetration_df must be DataFrame or Series with product id index/column")

    # If penetration looks like percent (max > 1), convert to fraction
    if pen.max() > 1.0:
        pen = pen / 100.0

    # Merge pairwise impacts with penetration of j
    merged = pairwise_impact_df[[product_i_col, product_j_col, impact_col]].copy()
    merged = merged.merge(
        pen.rename("penetration"),
        left_on=product_j_col,
        right_index=True,
        how="left"
    )

    # Fill missing penetration with tiny value
    merged['penetration'] = merged['penetration'].fillna(0.0).clip(lower=min_penetration)

    # Filter tiny impacts if requested
    merged = merged[merged[impact_col] >= min_impact].copy()

    # Compute contribution per pair (i->j)
    merged['contribution'] = merged[impact_col] * merged['penetration']

    # For topk method, keep top_k j per product_i by impact_col
    if method == "topk_weighted":
        merged = merged.sort_values([product_i_col, impact_col], ascending=[True, False])
        # rank within product_i
        merged['rank_within_i'] = merged.groupby(product_i_col)[impact_col].rank(method="first", ascending=False)
        merged = merged[merged['rank_within_i'] <= top_k].copy()
    elif method == "weighted_sum":
        pass
    elif method == "normalized":
        # we'll compute weighted sum then divide by sum of penetrations per i
        pass
    else:
        raise ValueError(f"Unknown method: {method}")

    # Aggregate per product_i
    agg = merged.groupby(product_i_col).agg(
        total_impact = ('contribution', 'sum'),
        total_weight = ('penetration', 'sum'),
        n_complements = (product_j_col, 'nunique'),
        mean_pair_impact = (impact_col, 'mean'),
    ).reset_index()

    # if normalized method, divide by total_weight (avoid divide by zero)
    if method == "normalized":
        agg['total_impact'] = agg['total_impact'] / agg['total_weight'].replace({0: np.nan})
        agg['total_impact'] = agg['total_impact'].fillna(0.0)

    # optional normalization across all products (0-1)
    if normalize:
        maxv = agg['total_impact'].max()
        if maxv > 0:
            agg['total_impact_norm'] = agg['total_impact'] / maxv
        else:
            agg['total_impact_norm'] = 0.0

    if return_details:
        return agg, merged
    else:
        return agg


product_info_df = pd.read_csv('../data/cleaned/product-features.csv')

totals, details = compute_total_impact(
    pairwise_impact_df=network_df,        # or original_df depending on which impact you want to aggregate
    penetration_df=product_info_df, # or series
    method="topk_weighted",
    top_k=5,
    normalize=True,
    return_details=True
)
totals.to_csv("../results3/complements-total-impact.csv", index=False)

# inspect top total impacts
print(totals.sort_values("total_impact", ascending=False).head(20))


   product_id  total_impact  total_weight  n_complements  mean_pair_impact  \
0       16696      0.000546      0.000801              5          0.706172   
5       43631      0.000448      0.000883              5          0.574459   
1       20738      0.000316      0.000873              5          0.363194   
3       27845      0.000113      0.000585              5          0.190997   
2       24852      0.000101      0.000515              5          0.196059   
4       32734      0.000060      0.000878              5          0.073045   

   total_impact_norm  
0           1.000000  
5           0.820555  
1           0.578623  
3           0.206769  
2           0.184757  
4           0.109187  
